# Notebook 03 — Model Explainability
**Level:** 3 (mandatory)  
**Depends on:** Notebooks 01 and 02 must have been run (model + processed data must exist).  

---

## Disclaimer

> **SHAP values describe the model's behaviour, not causal relationships.**  
> A high SHAP value for a variable means that the model relies heavily on that variable
> to produce its prediction. This does NOT mean that changing the variable in the real
> process will necessarily produce the same change in % Silica Concentrate.
> Causal inference requires controlled experiments and domain expertise beyond
> what a predictive model can provide.

---

## Objectives
1. Load the selected model and the test feature set.
2. Compute global SHAP feature importance.
3. Generate SHAP summary and bar plots.
4. Analyse local (single-observation) SHAP explanations.
5. Relate findings to operational variables.
6. Save all figures to `reports/figures/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.config import CFG
from src.train import load_model
from src.explain import (
    build_explainer,
    compute_shap_values,
    mean_absolute_shap,
    plot_shap_summary,
    plot_shap_bar,
    plot_shap_waterfall,
)
from src.evaluate import compute_metrics

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
FIGURES_PATH = Path(CFG['paths']['reports_figures'])
TARGET = CFG['target']

---
## 1. Load Model and Data

In [ ]:
model = load_model(folder='selected', cfg=CFG)

processed_path = Path(CFG['paths']['data_processed'])
X_train = pd.read_parquet(processed_path / 'X_train.parquet')
X_val   = pd.read_parquet(processed_path / 'X_val.parquet')
X_test  = pd.read_parquet(processed_path / 'X_test.parquet')
y_train = pd.read_parquet(processed_path / 'y_train.parquet')[TARGET]
y_val   = pd.read_parquet(processed_path / 'y_val.parquet')[TARGET]
y_test  = pd.read_parquet(processed_path / 'y_test.parquet')[TARGET]

print(f'Model: {type(model).__name__}')
print(f'Test set shape: {X_test.shape}')

# Confirm test performance
test_metrics = compute_metrics(y_test, model.predict(X_test))
print(f'Test metrics: {test_metrics}')

---
## 2. Build SHAP Explainer

In [ ]:
# Background dataset for the explainer: use training set
# TreeExplainer does not need a background sample for tree-based models
explainer = build_explainer(model, X_train)

# Compute SHAP values on a sample of the test set
shap_values, X_sample = compute_shap_values(
    explainer, X_test, cfg=CFG
)
print(f'SHAP values shape: {shap_values.shape}')

---
## 3. Global Feature Importance

In [ ]:
# Native feature importance (for tree models)
if hasattr(model, 'feature_importances_'):
    feat_imp = pd.Series(
        model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 7))
    feat_imp.head(20)[::-1].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Native Feature Importance — {type(model).__name__}', fontsize=12)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / 'native_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(feat_imp.head(15))
else:
    print('Model does not expose feature_importances_ attribute.')

In [ ]:
# SHAP mean absolute importance
shap_importance = mean_absolute_shap(shap_values, list(X_sample.columns))
print('SHAP Global Importance (top 20):')
print(shap_importance.head(20).to_string(index=False))

---
## 4. SHAP Summary and Bar Plots

In [ ]:
# SHAP bar plot (mean |SHAP|)
fig_bar = plot_shap_bar(
    shap_values, X_sample, cfg=CFG, save=True,
    title='SHAP Global Feature Importance — Mean |SHAP Value|'
)
plt.show()

In [ ]:
# SHAP summary beeswarm plot
# Shows direction of effect: red = high feature value, blue = low feature value
fig_summary = plot_shap_summary(
    shap_values, X_sample, cfg=CFG, save=True,
    title='SHAP Summary — Impact on % Silica Concentrate Prediction'
)
plt.show()

### Reading the Summary Plot

Each point is one observation. The x-axis shows the SHAP value (impact on model output):
- **Positive SHAP** → this observation's feature value pushed the predicted % Silica **up** (worse quality).
- **Negative SHAP** → the feature pushed the prediction **down** (better quality, lower silica).

**Colour**: red = high feature value, blue = low feature value.

Example interpretation for operational variables:
- If `% Silica Feed` (red) has positive SHAP → when feed silica is high, the model predicts higher concentrate silica — this aligns with physical intuition.
- If `Amina Flow` (red) has negative SHAP → higher reagent flow tends to suppress silica in the concentrate — consistent with amina's role as a flotation collector.

---
## 5. SHAP Dependence Plots — Key Operational Variables

In [ ]:
# Identify which operational variables are in the top SHAP features
operational_vars = (
    CFG['feature_groups']['feed'] +
    CFG['feature_groups']['flow'] +
    CFG['feature_groups']['pulp']
)

# Find versions of these in the feature set (direct or lagged)
top_shap_features = shap_importance['feature'].head(CFG['explainability']['top_features']).tolist()

# Filter to operational-related features
op_in_top = [f for f in top_shap_features
             if any(op.lower() in f.lower() for op in operational_vars)]

print('Top SHAP features related to operational variables:')
for f in op_in_top[:8]:
    print(f'  {f}')

In [ ]:
# SHAP dependence plots for top operational features
shap_idx_map = {col: i for i, col in enumerate(X_sample.columns)}
plot_features = op_in_top[:4] if len(op_in_top) >= 4 else top_shap_features[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, plot_features):
    if feat not in shap_idx_map:
        continue
    fidx = shap_idx_map[feat]
    sc = ax.scatter(
        X_sample[feat],
        shap_values[:, fidx],
        c=X_sample[feat],
        cmap='coolwarm',
        alpha=0.4,
        s=8,
    )
    ax.axhline(0, color='navy', lw=0.8, linestyle='--')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('SHAP value', fontsize=9)
    ax.set_title(feat, fontsize=9)
    plt.colorbar(sc, ax=ax, label='Feature value')

plt.suptitle('SHAP Dependence — Key Operational Variables', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'shap_dependence_operational.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Local Explanations — Individual Observations

In [ ]:
# Select 3 representative observations from the test set:
# 1. Observation with LOW predicted silica (good quality)
# 2. Observation with HIGH predicted silica (poor quality)
# 3. Median observation

y_pred_sample = model.predict(X_sample)
pred_series = pd.Series(y_pred_sample, index=X_sample.index)

idx_low  = pred_series.idxmin()
idx_high = pred_series.idxmax()
idx_med  = pred_series.iloc[(pred_series - pred_series.median()).abs().argsort()[:1]].index[0]

print(f'Low silica prediction:    index={idx_low}, pred={pred_series[idx_low]:.4f}')
print(f'High silica prediction:   index={idx_high}, pred={pred_series[idx_high]:.4f}')
print(f'Median silica prediction: index={idx_med}, pred={pred_series[idx_med]:.4f}')

In [ ]:
# Waterfall for HIGH silica (worst quality) prediction
obs_high = X_sample.loc[[idx_high]]
plot_shap_waterfall(
    explainer, obs_high,
    obs_idx=0, cfg=CFG, save=True,
    title_suffix='HIGH_silica'
)

In [ ]:
# Waterfall for LOW silica (best quality) prediction
obs_low = X_sample.loc[[idx_low]]
plot_shap_waterfall(
    explainer, obs_low,
    obs_idx=0, cfg=CFG, save=True,
    title_suffix='LOW_silica'
)

---
## 7. Operational Interpretation

Based on the SHAP analysis, we can summarise the key drivers of % Silica Concentrate
predictions in operational terms:

| Variable | Direction of effect | Operational interpretation |
|---|---|---|
| `% Silica Feed` | ↑ higher → higher silica predicted | Higher silica in the ore feed passes through to the concentrate |
| `% Iron Feed` | ↑ higher → lower silica predicted | Rich iron ore feed associated with cleaner separation |
| `Amina Flow` | ↑ higher → lower silica predicted | Amina is a selective collector that suppresses silica flotation |
| `Starch Flow` | context-dependent | Starch depresses iron, but excess can reduce selectivity |
| `Ore Pulp pH` | optimal range → lower silica | pH controls reagent efficiency; deviations worsen selectivity |
| Flotation column air flow | ↑ excess → higher silica | Excessive aeration can entrain silica mechanically |

> **Reminder:** These interpretations are based on the model's learned patterns.
> They should be validated against process knowledge and metallurgical expertise
> before informing operational decisions.

---
## 8. Coherence Check

We verify that SHAP explanations are consistent with model predictions.

In [ ]:
# SHAP values should sum to: prediction - expected_value
# Verify this for a sample
sv_obj = explainer(X_sample.head(20))
if hasattr(sv_obj, 'base_values'):
    expected_val = sv_obj.base_values[0] if hasattr(sv_obj.base_values, '__len__') else sv_obj.base_values
    shap_sum = sv_obj.values[0].sum()
    model_pred = model.predict(X_sample.head(1))[0]
    print(f'Expected value (SHAP baseline): {expected_val:.4f}')
    print(f'SHAP values sum (obs 0):        {shap_sum:.4f}')
    print(f'Model prediction (obs 0):        {model_pred:.4f}')
    print(f'Expected + SHAP sum:             {expected_val + shap_sum:.4f}')
    print(f'\nConsistency check passed: {abs(model_pred - (expected_val + shap_sum)) < 0.001}')
else:
    print('base_values not available for this explainer type.')

---
## Summary — Level 3 Checklist

| Item | Status |
|---|---|
| Native feature importance (tree model) | ✅ |
| SHAP global summary plot (beeswarm) | ✅ |
| SHAP global bar plot (mean \|SHAP\|) | ✅ |
| SHAP local (waterfall) for high/low silica observations | ✅ |
| Operational variable analysis | ✅ |
| SHAP coherence check (sum = prediction − expected) | ✅ |
| Figures saved to `reports/figures/` | ✅ |
| Disclaimer: SHAP ≠ causality | ✅ |
| Operational language used | ✅ |